In [1]:
import jax
import jax.numpy as jnp
import numpy as np
from numba import njit, prange

@jax.jit
def calc_rhs_jax(u, v, a, k) -> float:
    return -k*u*(u - a)*(u - 1) - u*v

@jax.jit
def calc_dv_jax(v, u, a, k, eps, mu1, mu2) -> float:
    dv = (- (eps + (mu1 * v) / (mu2 + u)) * (v + k * u * (u - a - 1.)))
    return dv


@jax.jit
def aliev_panfilov_jax(u, dt, v, a, k, eps, mu1, mu2):
    v += dt * calc_dv_jax(v, u, a, k, eps, mu1, mu2)
    rhs = dt * calc_rhs_jax(u, v, a, k)
    return v, rhs


@njit(cache=True)
def calc_rhs(u, v, a, k) -> float:
    return -k * u * (u - a) * (u - 1) - u * v

@njit(cache=True)
def calc_dv(v, u, a, k, eps, mu1, mu2) -> float:
    dv = (- (eps + (mu1 * v) / (mu2 + u)) * (v + k * u * (u - a - 1.)))
    return dv

@njit(parallel=True, cache=True)
def aliev_panfilov_numba(u, rhs, indexes, dt, v, a, k, eps, mu1, mu2):
    for i in prange(len(indexes)):
        ii = indexes[i]
        v.flat[ii] += dt * calc_dv(v.flat[ii], u.flat[ii], a, k, eps, mu1, mu2)
        rhs.flat[ii] = dt * calc_rhs(u.flat[ii], v.flat[ii], a, k)
    return v, rhs

In [4]:
import numpy as np

import jax.numpy as jnp

n = 100
mesh = np.ones((n, n, n), dtype=np.int8)
u = np.zeros((n, n, n), dtype=np.float64)
v = np.zeros((n, n, n), dtype=np.float64)
rhs = np.zeros((n, n, n), dtype=np.float64)

# mesh[np.random.rand(n, n, n) < 0.3] = 2
indexes = np.flatnonzero(mesh == 1)

u_jnp = jnp.zeros((n, n, n), dtype=jnp.float32)
v_jnp = jnp.zeros((n, n, n), dtype=jnp.float32)
mask = jnp.asarray(mesh == 1)

u[:5] = 1.0

dt = 0.01
a = 0.3
k = 8.0
eps = 0.002
mu1 = 0.2
mu2 = 0.3

aliev_panfilov_numba(u, rhs, indexes, dt, v, a, k, eps, mu1, mu2)
aliev_panfilov_jax(u_jnp, dt, v_jnp, a, k, eps, mu1, mu2)

%timeit aliev_panfilov_numba(u, rhs, indexes, dt, v, a, k, eps, mu1, mu2)
%timeit aliev_panfilov_jax(u_jnp, dt, v_jnp, a, k, eps, mu1, mu2)

118 μs ± 16.1 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
455 μs ± 3.54 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [5]:
%timeit np.asarray(u_jnp)
%timeit jnp.asarray(u)

832 ns ± 6 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)
452 μs ± 8.7 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


# Courtemance

In [6]:
import math
from numba import njit


@njit(cache=True)
def calc_rhs(ina, ik1, ito, ikur, ikr, iks, ical, ipca, inak, inaca, ibna, ibca):   
    return ina + ik1 + ito + ikur + ikr + iks + ical + ipca + inak + inaca + ibna + ibca


@njit(cache=True)
def calc_where(cond, x, y):
    if cond:
        return x
    return y


@njit(cache=True)
def calc_gating_variable(x, x_inf, tau_x,):
    return (x_inf - x) / tau_x

@njit(cache=True)
def calc_gating_variable_rush_larsen(x, x_inf, tau_x, dt, exp=math.exp):
    return x_inf - (x_inf - x)*exp(-dt/tau_x)


@njit(cache=True)
def calc_cmdn(cmdnmax, kmcmdn, cai):
    cmdn = cmdnmax*cai/(cai + kmcmdn)
    return cmdn


@njit(cache=True)
def calc_trpn(trpnmax, kmtrpn, cai):

    trpn = trpnmax*cai/(cai + kmtrpn)
    return trpn


@njit(cache=True)
def calc_csqn(csqnmax, kmcsqn, carel):
    csqn = csqnmax*carel/(carel + kmcsqn)
    return csqn


@njit(cache=True)
def calc_dnai(inak, inaca, ibna, ina, F, Vj):
    dnai = (-3*inak-3*inaca - ibna - ina)/(F*Vj)
    return dnai


@njit(cache=True)
def calc_dki(inak, ik1, ito, ikur, ikr, iks, ibk, F, Vj):
    dki = (2*inak - ik1 - ito - ikur - ikr - iks - ibk)/(F*Vj)
    return dki


@njit(cache=True)
def calc_dcai(cai, inaca, ipca, ical, ibca, iup, iupleak, irel, Vrel, Vup, trpnmax, kmtrpn, cmdnmax, kmcmdn, F, Vj): 
    B1 = (2*inaca - ipca - ical - ibca)/(2*F*Vj) + (Vup*(iupleak - iup) + irel*Vrel)/Vj
    B2 = 1 + (trpnmax*kmtrpn)/((cai + kmtrpn)**2) + (cmdnmax*kmcmdn)/((cai + kmcmdn)**2)
    dcai = B1/B2
    return dcai


@njit(cache=True)
def calc_dcaup(iup, iupleak, itr, Vrel, Vup):
    dcaup = iup - iupleak - itr*(Vrel/Vup)
    return dcaup


@njit(cache=True)
def calc_dcarel(carel, itr, irel, csqnmax, kmcsqn):
    dcarel = (itr - irel)/(1 + (csqnmax*kmcsqn)/((carel + kmcsqn)**2))
    return dcarel


@njit(cache=True)
def calc_equilibrum_potentials(nai, nao, ki, ko, cai, cao, R, T, F,
                               log=math.log, where=calc_where):
    ena = (R*T/F)*log(nao/nai)

    ek = (R*T/F)*log(ko/ki)

    safe_cai = where(cai < 1e-7, 1e-7, cai)

    eca = (R*T/(2*F))*log(cao/safe_cai)
    return ena, ek, eca


@njit(cache=True)
def calc_ina(u, m, h, j, gna, ena):
    ina = gna*(m**3)*h*j*(u - ena)
    return ina


@njit(cache=True)
def calc_gating_m(m, u, dt, exp=math.exp, where=calc_where):
    cond = u == -47.13
    denom = where(cond, 1., (1 - exp(-0.1 * (u + 47.13))))
    am = where(cond, 3.2, 0.32 * (u + 47.13) / denom)
    bm = 0.08*exp(-u/11)
    m_inf = am/(am + bm)
    tau_m = 1/(am + bm)
    m = calc_gating_variable_rush_larsen(m, m_inf, tau_m, dt, exp)

    return m


@njit(cache=True)
def calc_gating_h(h, u, dt, exp=math.exp, where=calc_where):
    cond = u >= -40
    ah = where(cond, 0, 0.135*exp(-(80 + u)/6.8))
    bh = where(cond, 1/(0.13*(1 + exp(-(u + 10.66)/11.1))),
               3.56*exp(0.079*u) + 310000*exp(0.35*u))
    h_inf = ah/(ah + bh)
    tau_h = 1/(ah + bh)
    h = calc_gating_variable_rush_larsen(h, h_inf, tau_h, dt, exp)
    return h


@njit(cache=True)
def calc_gating_j(j, u, dt, exp=math.exp, where=calc_where):
    cond = u >= -40
    aj = where(cond, 0,
               (-127140*exp(0.2444*u) - 0.00003474*exp(-0.04391*u)) *
               (u + 37.78)/(1 + exp(0.311*(u + 79.23))))
    bj = where(cond, 0.3*exp(-0.0000002535*u)/(1 + exp(-0.1*(u + 32))),
               0.1212*exp(-0.01052*u)/(1 + exp(-0.1378*(u + 40.14))))

    j_inf = aj/(aj + bj)
    tau_j = 1/(aj + bj)
    j = calc_gating_variable_rush_larsen(j, j_inf, tau_j, dt, exp)
    return j


@njit(cache=True)
def calc_ik1(u, gk1, ek, exp=math.exp):
    ik1 = gk1*(u - ek)/(1 + exp(0.07*(u + 80)))
    return ik1


@njit(cache=True)
def calc_ito(u, dt, kq10, oa, oi, gto, ek, exp=math.exp):
    ao = 0.65/(exp(-(u + 10)/8.5) + exp(-(u - 30)/59.0))
    bo = 0.65/(2.5 + exp((u + 82)/17.0))

    tau_o = 1/(kq10*(ao + bo))
    o_inf = 1/(1 + exp(-(u + 20.47)/17.54))

    aoi = 1/(18.53 + exp((u + 113.7)/10.95))
    boi = 1/(35.56 + exp(-(u + 1.26)/7.44))

    tau_oi = 1/(kq10*(aoi + boi))
    oi_inf = 1/(1 + exp((u + 43.1)/5.3))

    oa = calc_gating_variable_rush_larsen(oa, o_inf, tau_o, dt, exp)
    oi = calc_gating_variable_rush_larsen(oi, oi_inf, tau_oi, dt, exp)

    ito = gto*(oa**3)*oi*(u - ek)  

    return ito, oa, oi


@njit(cache=True)
def calc_ikur(u, dt, kq10, ua, ui, ek, gkur_coeff, exp=math.exp):
    gkur = 0.005 + 0.05/(1 + exp(-(u - 15)/13.0))

    aua = 0.65/(exp(-(u + 10)/8.5) + exp(-(u - 30)/59.0))
    bua = 0.65/(2.5 + exp((u + 82)/17.0))
    tau_ua = 1/(kq10*(aua + bua))
    ua_inf = 1/(1 + exp(-(u + 30.3)/9.6))
    aui = 1/(21 + exp(-(u - 185)/28.0))
    bui = exp((u - 158)/16.0)

    tau_ui = 1/(kq10*(aui + bui))
    ui_inf = 1/(1 + exp((u - 99.45)/27.48))

    ua = calc_gating_variable_rush_larsen(ua, ua_inf, tau_ua, dt, exp)
    ui = calc_gating_variable_rush_larsen(ui, ui_inf, tau_ui, dt, exp)

    ikur = gkur_coeff*gkur*(ua**3)*ui*(u - ek)

    return ikur, ua, ui


@njit(cache=True)
def calc_ikr(u, dt, xr, gkr, ek, exp=math.exp):
    gkr = 0.0294 # * np.sqrt(ko / 5.4)
    axr = 0.0003*(u + 14.1)/(1 - exp(-(u + 14.1)/5))
    bxr = 0.000073898*(u - 3.3328)/(exp((u - 3.3328)/5.1237) - 1)

    tau_xr = 1/(axr + bxr)
    xiinf = 1/(1 + exp(-(u + 14.1)/6.5))

    xr = calc_gating_variable_rush_larsen(xr, xiinf, tau_xr, dt, exp)

    ikr = (gkr*xr*(u - ek))/(1 + exp((u + 15)/22.4))

    return ikr, xr


@njit(cache=True)
def calc_iks(u, dt, xs, gks, ek, exp=math.exp, sqrt=math.sqrt):
    axs = 0.00004*(u - 19.9)/(1 - exp(-(u - 19.9)/17))
    bxs = 0.000035*(u - 19.9)/(exp((u - 19.9)/9) - 1)

    tau_xs = 1/(2*(axs + bxs))
    xs_inf = 1/sqrt(1 + exp(-(u - 19.9)/12.7))

    xs = calc_gating_variable_rush_larsen(xs, xs_inf, tau_xs, dt, exp)

    iks = gks*(xs**2)*(u - ek)

    return iks, xs


@njit(cache=True)
def calc_ical(u, dt, d, f, cai, gcal, fca, exp=math.exp):
    tau_d = (1 - exp(-(u + 10)/6.24))/(0.035*(u + 10)*
                                       (1 + exp(-(u + 10)/6.24)))
    iinf = 1/(1 + exp(-(u + 10)/8.0))

    tau_f = 9/(0.0197*exp(-(0.0337**2)*((u + 10)**2)) + 0.02)
    f_inf = 1/(1 + exp((u + 28)/6.9))

    tau_fca = 2
    fca_inf = 1/(1 + cai/0.00035)

    d   = calc_gating_variable_rush_larsen(d, iinf, tau_d, dt, exp)
    f   = calc_gating_variable_rush_larsen(f, f_inf, tau_f, dt, exp)
    fca = calc_gating_variable_rush_larsen(fca, fca_inf, tau_fca, dt, exp)

    ical = gcal*d*f*fca*(u - 65) 

    return ical, d, f, fca


@njit(cache=True)
def calc_inak(inakmax, nai, nao, ko, kmnai, kmko, F, u, R, T, exp=math.exp):
    s = (1/7.0)*(exp(nao/67.3) - 1)
    fnak = 1/(1 + 0.1245*exp(-0.1*(F*u)/(R*T)) + 0.0365*s*exp(-(F*u)/(R*T)))

    inak = inakmax*fnak*(1/(1 + (kmnai/nai)**1.5))*(ko/(ko + kmko))

    return inak


@njit(cache=True)
def calc_inaca(inacamax, nai, nao, cai, cao, kmnancx, kmcancx, ksatncx, F, u,
               R, T, exp=math.exp):
    gamma = 0.35

    # Exponential terms with clamping
    exp_term = exp(gamma * (F * u) / (R * T))
    exp_rev_term = exp((gamma - 1) * (F * u) / (R * T))

    # Numerator
    numerator = inacamax * (exp_term * nai**3 * cao - 
                            exp_rev_term * nao**3 * cai)

    # Denominator
    term1 = (kmnancx**3 + nao**3)  # (K_m,Na^3 + [Na+]_i^3)
    term2 = (kmcancx + cao)        # (K_m,Ca + [Ca2+]_o)
    term3 = (1 + ksatncx * exp_rev_term)  # (1 + k_sat * exp(...))

    denominator = term1 * term2 * term3

    # Calculate INaCa
    inaca = numerator / denominator

    return inaca


@njit(cache=True)
def calc_ibca(gcab, eca, u):
    ibca = gcab*(u - eca)
    return ibca


@njit(cache=True)
def calc_ibna(gnab, ena, u):
    ibna = gnab*(u - ena)
    return ibna

@njit(cache=True)
def calc_ipca(ipcamax, cai):
    ipca = ipcamax*cai/(cai + 0.0005)
    return ipca

@njit(cache=True)
def calc_irel(dt, urel, vrel, irel, wrel, ical, inaca, krel, carel, cai, u, F,
              Vrel, exp=math.exp):
    tau_u = 8

    Fn = 1e-12*Vrel*irel - ((5*1e-13)/F)*(0.5*ical - 0.2*inaca) 

    u_inf = 1/(1 + exp(-(Fn - 3.4175e-13)/13.67e-16))

    tau_v = 1.91 + 2.09/(1 + exp(-(Fn - 3.4175e-13)/13.67e-16))
    v_inf = 1 - 1/(1 + exp(-(Fn - 6.835e-14)/13.67e-16))

    tau_w = (6 * (1 - exp(-(u - 7.9) / 5.0)) /
             ((1 + 0.3 * exp(-(u - 7.9) / 5.0)) * (u - 7.9)))
    w_inf = 1 - 1/(1 + exp(-(u - 40)/17.0))

    urel = calc_gating_variable_rush_larsen(urel, u_inf, tau_u, dt, exp)
    vrel = calc_gating_variable_rush_larsen(vrel, v_inf, tau_v, dt, exp)
    wrel = calc_gating_variable_rush_larsen(wrel, w_inf, tau_w, dt, exp)

    irel = krel*(urel**2)*vrel*wrel*(carel - cai)

    return irel, urel, vrel, wrel

@njit(cache=True)
def calc_itr(caup, carel):
    tautr = 180
    itr = (caup - carel)/tautr
    return itr

@njit(cache=True)
def calc_iup(iupmax, cai, kup):
    iup = iupmax/(1 + (kup/cai))
    return iup

@njit(cache=True)
def calc_iupleak(caup, caupmax, iupmax):
    iupleak = (caup/caupmax)*iupmax
    return iupleak


@njit(parallel=True, fastmath=True, cache=True)
def courtemanche_numba(u, rhs, indexes, dt,
                 nai, ki, cai, caup, carel, m, h, j_, d, f, oa, oi, ua, ui, xs,
                 xr, fca, irel, vrel, urel, wrel,
                 gna, gnab, gk1, gkr, gks, gto, gcal, gcab, gkur_coeff, F, T,
                 R, Vc, Vj, Vup, Vrel, ibk, cao, nao, ko, caupmax, kup, kmnai,
                 kmko, kmnancx, kmcancx, ksatncx, kmcmdn, kmtrpn, kmcsqn,
                 trpnmax, cmdnmax, csqnmax, inacamax, inakmax, ipcamax, krel,
                 iupmax, kq10):


    for i in prange(indexes.shape[0]):
        ii = indexes[i]

        ena, ek, eca = calc_equilibrum_potentials(nai.flat[ii], nao,
                                                  ki.flat[ii], ko,
                                                  cai.flat[ii], cao, R, T, F)

        m.flat[ii] = calc_gating_m(m.flat[ii], u.flat[ii], dt)
        h.flat[ii] = calc_gating_h(h.flat[ii], u.flat[ii], dt)
        j_.flat[ii] = calc_gating_j(j_.flat[ii], u.flat[ii], dt)

        ina = calc_ina(u.flat[ii], m.flat[ii], h.flat[ii], j_.flat[ii],
                       gna, ena)
        ik1 = calc_ik1(u.flat[ii], gk1, ek)
        ito, oa.flat[ii], oi.flat[ii] = calc_ito(u.flat[ii], dt, kq10,
                                                   oa.flat[ii], oi.flat[ii],
                                                   gto, ek)
        ikur, ua.flat[ii], ui.flat[ii] = calc_ikur(u.flat[ii], dt, kq10,
                                                     ua.flat[ii],
                                                     ui.flat[ii], ek,
                                                     gkur_coeff)
        ikr, xr.flat[ii] = calc_ikr(u.flat[ii], dt, xr.flat[ii], gkr, ek)
        iks, xs.flat[ii] = calc_iks(u.flat[ii], dt, xs.flat[ii], gks, ek)
        ical, d.flat[ii], f.flat[ii], fca.flat[ii] = calc_ical(u.flat[ii],
                                                                  dt,
                                                                  d.flat[ii],
                                                                  f.flat[ii],
                                                                  cai.flat[ii],
                                                                  gcal,
                                                                  fca.flat[ii])
        inak = calc_inak(inakmax, nai.flat[ii], nao, ko, kmnai,
                         kmko, F, u.flat[ii], R, T)
        inaca = calc_inaca(inacamax, nai.flat[ii], nao, cai.flat[ii], cao,
                           kmnancx, kmcancx, ksatncx, F, u.flat[ii], R, T)
        ibca = calc_ibca(gcab, eca, u.flat[ii])
        ibna = calc_ibna(gnab, ena, u.flat[ii])
        ipca = calc_ipca(ipcamax, cai.flat[ii])
        irel.flat[ii], urel.flat[ii], vrel.flat[ii], wrel.flat[ii] = calc_irel(
            dt, urel.flat[ii], vrel.flat[ii], irel.flat[ii], wrel.flat[ii],
            ical, inaca, krel, carel.flat[ii], cai.flat[ii], u.flat[ii],
            F, Vrel)
        itr = calc_itr(caup.flat[ii], carel.flat[ii])
        iup = calc_iup(iupmax, cai.flat[ii], kup)
        iupleak = calc_iupleak(caup.flat[ii], caupmax, iupmax)

        caup.flat[ii] += dt * calc_dcaup(iup, iupleak, itr, Vrel, Vup)
        nai.flat[ii] += dt * calc_dnai(inak, inaca, ibna, ina, F, Vj)

        ki.flat[ii] += dt * calc_dki(inak, ik1, ito, ikur, ikr, iks, ibk, F,
                                      Vj)
        cai.flat[ii] += dt * calc_dcai(cai.flat[ii], inaca, ipca, ical, ibca,
                                        iup, iupleak, irel.flat[ii], Vrel,
                                        Vup, trpnmax, kmtrpn, cmdnmax, kmcmdn,
                                        F, Vj)

        carel.flat[ii] += dt * calc_dcarel(carel.flat[ii], itr,
                                            irel.flat[ii], csqnmax, kmcsqn)

        rhs.flat[ii] = dt*(-calc_rhs(ina, ik1, ito, ikur, ikr, iks, ical,
                                      ipca, inak, inaca, ibna, ibca))


In [7]:
gna = 7.8
gnab = 0.000674
gk1 = 0.09
gkr = 0.0294
gks = 0.129
gto = 0.1652
gcal = 0.1238
gcab = 0.00113
gkur_coeff = 1.0
F = 96485.0
T = 310.0
R = 8314.0
Vc = 20100.0
Vj = 20100.0 * 0.68
Vup = 20100.0 * 0.68 * 0.06 * 0.92
Vrel = 20100.0 * 0.68 * 0.06 * 0.08
ibk = 0.0
cao = 1.8
nao = 140.0
ko = 5.4
caupmax = 15.0
kup = 0.00092
kmnai = 10.0
kmko = 1.5
kmnancx = 87.5
kmcancx = 1.38
ksatncx = 0.1
kmcmdn = 0.00238
kmtrpn = 0.0005
kmcsqn = 0.8
trpnmax = 0.07
cmdnmax = 0.05
csqnmax = 10.0
inacamax = 1600.0
inakmax = 0.6
ipcamax = 0.275
krel = 30.0
iupmax = 0.005
kq10 = 3.0


n = 100
mesh = np.ones((n, n, n), dtype=np.int8)
u = -84.5 * np.ones((n, n, n), dtype=np.float64)
nai = 11.2 * np.ones((n, n, n), dtype=np.float64)
ki = 139.0 * np.ones((n, n, n), dtype=np.float64)
cai = 0.000102 * np.ones((n, n, n), dtype=np.float64)
caup = 1.6 * np.ones((n, n, n), dtype=np.float64)
carel = 1.1 * np.ones((n, n, n), dtype=np.float64)
m = 0.00291 * np.ones((n, n, n), dtype=np.float64)
h = 0.965 * np.ones((n, n, n), dtype=np.float64)
j_ = 0.978 * np.ones((n, n, n), dtype=np.float64)
d = 0.000137 * np.ones((n, n, n), dtype=np.float64)
f = 0.999837 * np.ones((n, n, n), dtype=np.float64)
oa = 0.000592 * np.ones((n, n, n), dtype=np.float64)
oi = 0.9992 * np.ones((n, n, n), dtype=np.float64)
ua = 0.003519 * np.ones((n, n, n), dtype=np.float64)
ui = 0.9987 * np.ones((n, n, n), dtype=np.float64)
xs = 0.0187 * np.ones((n, n, n), dtype=np.float64)
xr = 0.0000329 * np.ones((n, n, n), dtype=np.float64)
fca = 0.775 * np.ones((n, n, n), dtype=np.float64)
irel = 0.0 * np.ones((n, n, n), dtype=np.float64)
vrel = 1.0 * np.ones((n, n, n), dtype=np.float64)
urel = 0.0 * np.ones((n, n, n), dtype=np.float64)
wrel = 0.9 * np.ones((n, n, n), dtype=np.float64)

indexes = np.flatnonzero(mesh == 1)

%timeit courtemanche_numba(u, rhs, indexes, dt, nai, ki, cai, caup, carel, m, h, j_, d, f, oa, oi, ua, ui, xs, xr, fca, irel, vrel, urel, wrel, gna, gnab, gk1, gkr, gks, gto, gcal, gcab, gkur_coeff, F, T, R, Vc, Vj, Vup, Vrel, ibk, cao, nao, ko, caupmax, kup, kmnai, kmko, kmnancx, kmcancx, ksatncx, kmcmdn, kmtrpn, kmcsqn, trpnmax, cmdnmax, csqnmax, inacamax, inakmax, ipcamax, krel, iupmax, kq10)

8.13 ms ± 798 μs per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [9]:
import math
import jax
import jax.numpy as jnp
from functools import partial


@partial(jax.jit, static_argnames=())
def calc_rhs(ina, ik1, ito, ikur, ikr, iks, ical, ipca, inak, inaca, ibna, ibca):   
    return ina + ik1 + ito + ikur + ikr + iks + ical + ipca + inak + inaca + ibna + ibca


@partial(jax.jit, static_argnames=())
def calc_where(cond, x, y):
    return jnp.where(cond, x, y)


@partial(jax.jit, static_argnames=())
def calc_gating_variable(x, x_inf, tau_x,):
    return (x_inf - x) / tau_x

@partial(jax.jit, static_argnames=('exp'))
def calc_gating_variable_rush_larsen(x, x_inf, tau_x, dt, exp=math.exp):
    return x_inf - (x_inf - x)*exp(-dt/tau_x)


@partial(jax.jit, static_argnames=())
def calc_cmdn(cmdnmax, kmcmdn, cai):
    cmdn = cmdnmax*cai/(cai + kmcmdn)
    return cmdn


@partial(jax.jit, static_argnames=())
def calc_trpn(trpnmax, kmtrpn, cai):

    trpn = trpnmax*cai/(cai + kmtrpn)
    return trpn


@partial(jax.jit, static_argnames=())
def calc_csqn(csqnmax, kmcsqn, carel):
    csqn = csqnmax*carel/(carel + kmcsqn)
    return csqn


@partial(jax.jit, static_argnames=())
def calc_dnai(inak, inaca, ibna, ina, F, Vj):
    dnai = (-3*inak-3*inaca - ibna - ina)/(F*Vj)
    return dnai


@partial(jax.jit, static_argnames=())
def calc_dki(inak, ik1, ito, ikur, ikr, iks, ibk, F, Vj):
    dki = (2*inak - ik1 - ito - ikur - ikr - iks - ibk)/(F*Vj)
    return dki


@partial(jax.jit, static_argnames=())
def calc_dcai(cai, inaca, ipca, ical, ibca, iup, iupleak, irel, Vrel, Vup, trpnmax, kmtrpn, cmdnmax, kmcmdn, F, Vj): 
    B1 = (2*inaca - ipca - ical - ibca)/(2*F*Vj) + (Vup*(iupleak - iup) + irel*Vrel)/Vj
    B2 = 1 + (trpnmax*kmtrpn)/((cai + kmtrpn)**2) + (cmdnmax*kmcmdn)/((cai + kmcmdn)**2)
    dcai = B1/B2
    return dcai


@partial(jax.jit, static_argnames=())
def calc_dcaup(iup, iupleak, itr, Vrel, Vup):
    dcaup = iup - iupleak - itr*(Vrel/Vup)
    return dcaup


@partial(jax.jit, static_argnames=())
def calc_dcarel(carel, itr, irel, csqnmax, kmcsqn):
    dcarel = (itr - irel)/(1 + (csqnmax*kmcsqn)/((carel + kmcsqn)**2))
    return dcarel


@partial(jax.jit, static_argnames=('log', 'where'))
def calc_equilibrum_potentials(nai, nao, ki, ko, cai, cao, R, T, F,
                               log=math.log, where=calc_where):
    ena = (R*T/F)*log(nao/nai)

    ek = (R*T/F)*log(ko/ki)

    safe_cai = where(cai < 1e-7, 1e-7, cai)

    eca = (R*T/(2*F))*log(cao/safe_cai)
    return ena, ek, eca


@partial(jax.jit, static_argnames=())
def calc_ina(u, m, h, j, gna, ena):
    ina = gna*(m**3)*h*j*(u - ena)
    return ina


@partial(jax.jit, static_argnames=('exp', 'where'))
def calc_gating_m(m, u, dt, exp=math.exp, where=calc_where):
    cond = u == -47.13
    denom = where(cond, 1., (1 - exp(-0.1 * (u + 47.13))))
    am = where(cond, 3.2, 0.32 * (u + 47.13) / denom)
    bm = 0.08*exp(-u/11)
    m_inf = am/(am + bm)
    tau_m = 1/(am + bm)
    m = calc_gating_variable_rush_larsen(m, m_inf, tau_m, dt, exp)

    return m


@partial(jax.jit, static_argnames=('exp', 'where'))
def calc_gating_h(h, u, dt, exp=math.exp, where=calc_where):
    cond = u >= -40
    ah = where(cond, 0, 0.135*exp(-(80 + u)/6.8))
    bh = where(cond, 1/(0.13*(1 + exp(-(u + 10.66)/11.1))),
               3.56*exp(0.079*u) + 310000*exp(0.35*u))
    h_inf = ah/(ah + bh)
    tau_h = 1/(ah + bh)
    h = calc_gating_variable_rush_larsen(h, h_inf, tau_h, dt, exp)
    return h


@partial(jax.jit, static_argnames=('exp', 'where'))
def calc_gating_j(j, u, dt, exp=math.exp, where=calc_where):
    cond = u >= -40
    aj = where(cond, 0,
               (-127140*exp(0.2444*u) - 0.00003474*exp(-0.04391*u)) *
               (u + 37.78)/(1 + exp(0.311*(u + 79.23))))
    bj = where(cond, 0.3*exp(-0.0000002535*u)/(1 + exp(-0.1*(u + 32))),
               0.1212*exp(-0.01052*u)/(1 + exp(-0.1378*(u + 40.14))))

    j_inf = aj/(aj + bj)
    tau_j = 1/(aj + bj)
    j = calc_gating_variable_rush_larsen(j, j_inf, tau_j, dt, exp)
    return j


@partial(jax.jit, static_argnames=('exp'))
def calc_ik1(u, gk1, ek, exp=math.exp):
    ik1 = gk1*(u - ek)/(1 + exp(0.07*(u + 80)))
    return ik1


@partial(jax.jit, static_argnames=('exp'))
def calc_ito(u, dt, kq10, oa, oi, gto, ek, exp=math.exp):
    ao = 0.65/(exp(-(u + 10)/8.5) + exp(-(u - 30)/59.0))
    bo = 0.65/(2.5 + exp((u + 82)/17.0))

    tau_o = 1/(kq10*(ao + bo))
    o_inf = 1/(1 + exp(-(u + 20.47)/17.54))

    aoi = 1/(18.53 + exp((u + 113.7)/10.95))
    boi = 1/(35.56 + exp(-(u + 1.26)/7.44))

    tau_oi = 1/(kq10*(aoi + boi))
    oi_inf = 1/(1 + exp((u + 43.1)/5.3))

    oa = calc_gating_variable_rush_larsen(oa, o_inf, tau_o, dt, exp)
    oi = calc_gating_variable_rush_larsen(oi, oi_inf, tau_oi, dt, exp)

    ito = gto*(oa**3)*oi*(u - ek)  

    return ito, oa, oi


@partial(jax.jit, static_argnames=('exp'))
def calc_ikur(u, dt, kq10, ua, ui, ek, gkur_coeff, exp=math.exp):
    gkur = 0.005 + 0.05/(1 + exp(-(u - 15)/13.0))

    aua = 0.65/(exp(-(u + 10)/8.5) + exp(-(u - 30)/59.0))
    bua = 0.65/(2.5 + exp((u + 82)/17.0))
    tau_ua = 1/(kq10*(aua + bua))
    ua_inf = 1/(1 + exp(-(u + 30.3)/9.6))
    aui = 1/(21 + exp(-(u - 185)/28.0))
    bui = exp((u - 158)/16.0)

    tau_ui = 1/(kq10*(aui + bui))
    ui_inf = 1/(1 + exp((u - 99.45)/27.48))

    ua = calc_gating_variable_rush_larsen(ua, ua_inf, tau_ua, dt, exp)
    ui = calc_gating_variable_rush_larsen(ui, ui_inf, tau_ui, dt, exp)

    ikur = gkur_coeff*gkur*(ua**3)*ui*(u - ek)

    return ikur, ua, ui


@partial(jax.jit, static_argnames=('exp'))
def calc_ikr(u, dt, xr, gkr, ek, exp=math.exp):
    gkr = 0.0294 # * np.sqrt(ko / 5.4)
    axr = 0.0003*(u + 14.1)/(1 - exp(-(u + 14.1)/5))
    bxr = 0.000073898*(u - 3.3328)/(exp((u - 3.3328)/5.1237) - 1)

    tau_xr = 1/(axr + bxr)
    xiinf = 1/(1 + exp(-(u + 14.1)/6.5))

    xr = calc_gating_variable_rush_larsen(xr, xiinf, tau_xr, dt, exp)

    ikr = (gkr*xr*(u - ek))/(1 + exp((u + 15)/22.4))

    return ikr, xr


@partial(jax.jit, static_argnames=('exp', 'sqrt'))
def calc_iks(u, dt, xs, gks, ek, exp=math.exp, sqrt=math.sqrt):
    axs = 0.00004*(u - 19.9)/(1 - exp(-(u - 19.9)/17))
    bxs = 0.000035*(u - 19.9)/(exp((u - 19.9)/9) - 1)

    tau_xs = 1/(2*(axs + bxs))
    xs_inf = 1/sqrt(1 + exp(-(u - 19.9)/12.7))

    xs = calc_gating_variable_rush_larsen(xs, xs_inf, tau_xs, dt, exp)

    iks = gks*(xs**2)*(u - ek)

    return iks, xs


@partial(jax.jit, static_argnames=('exp'))
def calc_ical(u, dt, d, f, cai, gcal, fca, exp=math.exp):
    tau_d = (1 - exp(-(u + 10)/6.24))/(0.035*(u + 10)*
                                       (1 + exp(-(u + 10)/6.24)))
    iinf = 1/(1 + exp(-(u + 10)/8.0))

    tau_f = 9/(0.0197*exp(-(0.0337**2)*((u + 10)**2)) + 0.02)
    f_inf = 1/(1 + exp((u + 28)/6.9))

    tau_fca = 2
    fca_inf = 1/(1 + cai/0.00035)

    d   = calc_gating_variable_rush_larsen(d, iinf, tau_d, dt, exp)
    f   = calc_gating_variable_rush_larsen(f, f_inf, tau_f, dt, exp)
    fca = calc_gating_variable_rush_larsen(fca, fca_inf, tau_fca, dt, exp)

    ical = gcal*d*f*fca*(u - 65) 

    return ical, d, f, fca


@partial(jax.jit, static_argnames=('exp'))
def calc_inak(inakmax, nai, nao, ko, kmnai, kmko, F, u, R, T, exp=math.exp):
    s = (1/7.0)*(exp(nao/67.3) - 1)
    fnak = 1/(1 + 0.1245*exp(-0.1*(F*u)/(R*T)) + 0.0365*s*exp(-(F*u)/(R*T)))

    inak = inakmax*fnak*(1/(1 + (kmnai/nai)**1.5))*(ko/(ko + kmko))

    return inak


@partial(jax.jit, static_argnames=('exp'))
def calc_inaca(inacamax, nai, nao, cai, cao, kmnancx, kmcancx, ksatncx, F, u,
               R, T, exp=math.exp):
    gamma = 0.35

    # Exponential terms with clamping
    exp_term = exp(gamma * (F * u) / (R * T))
    exp_rev_term = exp((gamma - 1) * (F * u) / (R * T))

    # Numerator
    numerator = inacamax * (exp_term * nai**3 * cao - 
                            exp_rev_term * nao**3 * cai)

    # Denominator
    term1 = (kmnancx**3 + nao**3)  # (K_m,Na^3 + [Na+]_i^3)
    term2 = (kmcancx + cao)        # (K_m,Ca + [Ca2+]_o)
    term3 = (1 + ksatncx * exp_rev_term)  # (1 + k_sat * exp(...))

    denominator = term1 * term2 * term3

    # Calculate INaCa
    inaca = numerator / denominator

    return inaca


@partial(jax.jit, static_argnames=())
def calc_ibca(gcab, eca, u):
    ibca = gcab*(u - eca)
    return ibca


@partial(jax.jit, static_argnames=())
def calc_ibna(gnab, ena, u):
    ibna = gnab*(u - ena)
    return ibna

@partial(jax.jit, static_argnames=())
def calc_ipca(ipcamax, cai):
    ipca = ipcamax*cai/(cai + 0.0005)
    return ipca

@partial(jax.jit, static_argnames=('exp'))
def calc_irel(dt, urel, vrel, irel, wrel, ical, inaca, krel, carel, cai, u, F,
              Vrel, exp=math.exp):
    tau_u = 8

    Fn = 1e-12*Vrel*irel - ((5*1e-13)/F)*(0.5*ical - 0.2*inaca) 

    u_inf = 1/(1 + exp(-(Fn - 3.4175e-13)/13.67e-16))

    tau_v = 1.91 + 2.09/(1 + exp(-(Fn - 3.4175e-13)/13.67e-16))
    v_inf = 1 - 1/(1 + exp(-(Fn - 6.835e-14)/13.67e-16))

    tau_w = (6 * (1 - exp(-(u - 7.9) / 5.0)) /
             ((1 + 0.3 * exp(-(u - 7.9) / 5.0)) * (u - 7.9)))
    w_inf = 1 - 1/(1 + exp(-(u - 40)/17.0))

    urel = calc_gating_variable_rush_larsen(urel, u_inf, tau_u, dt, exp)
    vrel = calc_gating_variable_rush_larsen(vrel, v_inf, tau_v, dt, exp)
    wrel = calc_gating_variable_rush_larsen(wrel, w_inf, tau_w, dt, exp)

    irel = krel*(urel**2)*vrel*wrel*(carel - cai)

    return irel, urel, vrel, wrel

@partial(jax.jit, static_argnames=())
def calc_itr(caup, carel):
    tautr = 180
    itr = (caup - carel)/tautr
    return itr

@partial(jax.jit, static_argnames=())
def calc_iup(iupmax, cai, kup):
    iup = iupmax/(1 + (kup/cai))
    return iup

@partial(jax.jit, static_argnames=())
def calc_iupleak(caup, caupmax, iupmax):
    iupleak = (caup/caupmax)*iupmax
    return iupleak


@jax.jit
def courtemanche_jax(u, dt,
                 nai, ki, cai, caup, carel, m, h, j_, d, f, oa, oi, ua, ui, xs,
                 xr, fca, irel, vrel, urel, wrel,
                 gna, gnab, gk1, gkr, gks, gto, gcal, gcab, gkur_coeff, F, T,
                 R, Vc, Vj, Vup, Vrel, ibk, cao, nao, ko, caupmax, kup, kmnai,
                 kmko, kmnancx, kmcancx, ksatncx, kmcmdn, kmtrpn, kmcsqn,
                 trpnmax, cmdnmax, csqnmax, inacamax, inakmax, ipcamax, krel,
                 iupmax, kq10):

    ena, ek, eca = calc_equilibrum_potentials(nai, nao,
                                                ki, ko,
                                                cai, cao, R, T, F, log=jnp.log, where=calc_where)

    m = calc_gating_m(m, u, dt, exp=jnp.exp, where=calc_where)
    h = calc_gating_h(h, u, dt, exp=jnp.exp, where=calc_where)
    j_ = calc_gating_j(j_, u, dt, exp=jnp.exp, where=calc_where)

    ina = calc_ina(u, m, h, j_,
                    gna, ena)
    ik1 = calc_ik1(u, gk1, ek, exp=jnp.exp)
    ito, oa, oi = calc_ito(u, dt, kq10, oa, oi, gto, ek, exp=jnp.exp)
    ikur, ua, ui = calc_ikur(u, dt, kq10, ua, ui, ek, gkur_coeff, exp=jnp.exp)
    ikr, xr = calc_ikr(u, dt, xr, gkr, ek, exp=jnp.exp)
    iks, xs = calc_iks(u, dt, xs, gks, ek, exp=jnp.exp, sqrt=jnp.sqrt)
    ical, d, f, fca = calc_ical(u, dt, d, f, cai, gcal, fca, exp=jnp.exp)
    inak = calc_inak(inakmax, nai, nao, ko, kmnai, kmko, F, u, R, T, exp=jnp.exp)
    inaca = calc_inaca(inacamax, nai, nao, cai, cao, kmnancx, kmcancx, ksatncx, F, u, R, T, exp=jnp.exp)
    ibca = calc_ibca(gcab, eca, u)
    ibna = calc_ibna(gnab, ena, u)
    ipca = calc_ipca(ipcamax, cai)
    irel, urel, vrel, wrel = calc_irel(dt, urel, vrel, irel, wrel, ical, inaca,
                                       krel, carel, cai, u, F, Vrel, exp=jnp.exp)
    itr = calc_itr(caup, carel)
    iup = calc_iup(iupmax, cai, kup)
    iupleak = calc_iupleak(caup, caupmax, iupmax)

    caup += dt * calc_dcaup(iup, iupleak, itr, Vrel, Vup)
    nai += dt * calc_dnai(inak, inaca, ibna, ina, F, Vj)

    ki += dt * calc_dki(inak, ik1, ito, ikur, ikr, iks, ibk, F,
                                    Vj)
    cai += dt * calc_dcai(cai, inaca, ipca, ical, ibca,
                                    iup, iupleak, irel, Vrel,
                                    Vup, trpnmax, kmtrpn, cmdnmax, kmcmdn,
                                    F, Vj)

    carel += dt * calc_dcarel(carel, itr,
                                        irel, csqnmax, kmcsqn)

    rhs = dt*(-calc_rhs(ina, ik1, ito, ikur, ikr, iks, ical,
                                    ipca, inak, inaca, ibna, ibca))
    return rhs, u, nai, ki, cai, caup, carel, m, h, j_, d, f, oa, oi, ua, ui, xs, xr, fca, irel, vrel, urel, wrel


In [10]:
# jax.config.update("jax_enable_x64", True)

n = 100
mesh = jnp.ones((n, n, n), dtype=jnp.int8)
u = -84.5 * jnp.ones((n, n, n), dtype=jnp.float32)
nai = 11.2 * jnp.ones((n, n, n), dtype=jnp.float32)
ki = 139.0 * jnp.ones((n, n, n), dtype=jnp.float32)
cai = 0.000102 * jnp.ones((n, n, n), dtype=jnp.float32)
caup = 1.6 * jnp.ones((n, n, n), dtype=jnp.float32)
carel = 1.1 * jnp.ones((n, n, n), dtype=jnp.float32)
m = 0.00291 * jnp.ones((n, n, n), dtype=jnp.float32)
h = 0.965 * jnp.ones((n, n, n), dtype=jnp.float32)
j_ = 0.978 * jnp.ones((n, n, n), dtype=jnp.float32)
d = 0.000137 * jnp.ones((n, n, n), dtype=jnp.float32)
f = 0.999837 * jnp.ones((n, n, n), dtype=jnp.float32)
oa = 0.000592 * jnp.ones((n, n, n), dtype=jnp.float32)
oi = 0.9992 * jnp.ones((n, n, n), dtype=jnp.float32)
ua = 0.003519 * jnp.ones((n, n, n), dtype=jnp.float32)
ui = 0.9987 * jnp.ones((n, n, n), dtype=jnp.float32)
xs = 0.0187 * jnp.ones((n, n, n), dtype=jnp.float32)
xr = 0.0000329 * jnp.ones((n, n, n), dtype=jnp.float32)
fca = 0.775 * jnp.ones((n, n, n), dtype=jnp.float32)
irel = 0.0 * jnp.ones((n, n, n), dtype=jnp.float32)
vrel = 1.0 * jnp.ones((n, n, n), dtype=jnp.float32)
urel = 0.0 * jnp.ones((n, n, n), dtype=jnp.float32)
wrel = 0.9 * jnp.ones((n, n, n), dtype=jnp.float32)

# for i in range(10000):
courtemanche_jax(u, dt, nai, ki, cai, caup, carel, m, h, j_, d, f, oa, oi, ua, ui, xs, xr, fca, irel, vrel, urel, wrel, gna, gnab, gk1, gkr, gks, gto, gcal, gcab, gkur_coeff, F, T, R, Vc, Vj, Vup, Vrel, ibk, cao, nao, ko, caupmax, kup, kmnai, kmko, kmnancx, kmcancx, ksatncx, kmcmdn, kmtrpn, kmcsqn, trpnmax, cmdnmax, csqnmax, inacamax, inakmax, ipcamax, krel, iupmax, kq10)
%timeit courtemanche_jax(u, dt, nai, ki, cai, caup, carel, m, h, j_, d, f, oa, oi, ua, ui, xs, xr, fca, irel, vrel, urel, wrel, gna, gnab, gk1, gkr, gks, gto, gcal, gcab, gkur_coeff, F, T, R, Vc, Vj, Vup, Vrel, ibk, cao, nao, ko, caupmax, kup, kmnai, kmko, kmnancx, kmcancx, ksatncx, kmcmdn, kmtrpn, kmcsqn, trpnmax, cmdnmax, csqnmax, inacamax, inakmax, ipcamax, krel, iupmax, kq10)

18.7 ms ± 678 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
